# ML + LLM Pipeline: Combining ML with LLMs

This is one of the most powerful patterns in production AI:
- Use **ML (scikit-learn)** for scoring and classification (fast, cheap, deterministic)
- Use **LLMs (GPT-4o)** for reasoning about those scores (flexible, intelligent)

**The KEY IDEA:**
- ML models produce **EVIDENCE** (scores, classifications, probabilities).
- LLMs **REASON** about that evidence to make decisions and write explanations.
- The LLM is not guessing -- it's analyzing real data.

**Pipeline:**
- **Stage 1:** ML model scores/classifies the data (scikit-learn)
- **Stage 2:** LLM reasons about the ML results and makes a recommendation

**Example use case:** Lead scoring for a real estate agent.
- ML model scores leads based on behavioral features
- LLM explains the score and recommends specific actions

## Install Dependencies

In [ ]:
!pip install openai scikit-learn numpy

## Imports and Configuration

In [ ]:
import json
import os

from openai import OpenAI
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler


client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
MODEL = "gpt-4o"

## Stage 1: ML Model -- Lead Scoring

This stage uses a traditional ML model (Random Forest) to score leads.
It's fast, cheap, and deterministic. Same input = same output every time.

**WHY ML AND NOT AN LLM?**
- Scoring 10,000 leads with ML: ~1 second, ~$0.00
- Scoring 10,000 leads with LLM: ~30 minutes, ~$50.00
- ML doesn't hallucinate scores -- they're mathematically computed

### Train the Lead Scoring Model

In production, you'd train this on real historical data (leads that
converted vs. leads that didn't). Here we use synthetic data to
demonstrate the pattern.

**Features:**
- `days_since_last_contact`: How recently we talked to them
- `website_visits_this_week`: How active they are on our site
- `email_open_rate`: How engaged they are with our emails
- `property_views`: How many listings they've looked at
- `days_on_market`: How long they've been searching
- `price_range_match`: How well our inventory matches their budget (0-1)

In [ ]:
def train_lead_scoring_model():
    """
    Train a simple lead scoring model using scikit-learn.

    In production, you'd train this on real historical data (leads that
    converted vs. leads that didn't). Here we use synthetic data to
    demonstrate the pattern.

    Features:
        - days_since_last_contact: How recently we talked to them
        - website_visits_this_week: How active they are on our site
        - email_open_rate: How engaged they are with our emails
        - property_views: How many listings they've looked at
        - days_on_market: How long they've been searching
        - price_range_match: How well our inventory matches their budget (0-1)
    """

    # --- Synthetic training data ---
    # In production, this would be real historical lead data.
    # Columns: [days_since_contact, website_visits, email_open_rate,
    #           property_views, days_on_market, price_range_match]
    np.random.seed(42)

    # Generate 200 synthetic leads
    n_samples = 200

    # "Hot" leads (label = 1): recent contact, high engagement
    hot_leads = np.column_stack([
        np.random.randint(1, 7, n_samples // 2),        # days_since_contact: 1-7
        np.random.randint(5, 15, n_samples // 2),       # website_visits: 5-15
        np.random.uniform(0.5, 0.9, n_samples // 2),    # email_open_rate: 0.5-0.9
        np.random.randint(8, 25, n_samples // 2),       # property_views: 8-25
        np.random.randint(7, 60, n_samples // 2),       # days_on_market: 7-60
        np.random.uniform(0.6, 1.0, n_samples // 2),    # price_range_match: 0.6-1.0
    ])

    # "Cold" leads (label = 0): old contact, low engagement
    cold_leads = np.column_stack([
        np.random.randint(14, 90, n_samples // 2),      # days_since_contact: 14-90
        np.random.randint(0, 3, n_samples // 2),        # website_visits: 0-3
        np.random.uniform(0.0, 0.3, n_samples // 2),    # email_open_rate: 0.0-0.3
        np.random.randint(0, 5, n_samples // 2),        # property_views: 0-5
        np.random.randint(60, 180, n_samples // 2),     # days_on_market: 60-180
        np.random.uniform(0.0, 0.4, n_samples // 2),    # price_range_match: 0.0-0.4
    ])

    X = np.vstack([hot_leads, cold_leads])
    y = np.array([1] * (n_samples // 2) + [0] * (n_samples // 2))

    # Shuffle the data
    shuffle_idx = np.random.permutation(n_samples)
    X = X[shuffle_idx]
    y = y[shuffle_idx]

    # Train the model
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_scaled, y)

    print("ML Model trained successfully.")
    print(f"Training accuracy: {model.score(X_scaled, y):.2%}")

    return model, scaler

### Score a Lead with the ML Model

Score a single lead using the trained ML model. This produces **EVIDENCE**
that the LLM will reason about in the next stage.

In [ ]:
def score_lead_with_ml(model, scaler, lead_features: dict) -> dict:
    """
    Score a single lead using the trained ML model.

    Input: A dict of lead features
    Output: A dict with the original features PLUS ML-generated scores

    This is the ML stage of the pipeline. It produces EVIDENCE that
    the LLM will reason about in the next stage.
    """

    print("=" * 60)
    print("STAGE 1: ML Lead Scoring")
    print("=" * 60)

    # Extract features in the correct order
    feature_names = [
        "days_since_last_contact",
        "website_visits_this_week",
        "email_open_rate",
        "property_views",
        "days_on_market",
        "price_range_match"
    ]

    feature_values = np.array([[lead_features[f] for f in feature_names]])
    feature_values_scaled = scaler.transform(feature_values)

    # Get prediction and probability
    prediction = model.predict(feature_values_scaled)[0]
    probability = model.predict_proba(feature_values_scaled)[0]

    # Get feature importances (which features mattered most for this prediction)
    importances = model.feature_importances_

    # Build the ML output — this is the EVIDENCE for the LLM
    ml_results = {
        "lead_info": lead_features,
        "ml_scores": {
            "hot_lead_probability": round(float(probability[1]), 3),
            "cold_lead_probability": round(float(probability[0]), 3),
            "classification": "HOT" if prediction == 1 else "COLD",
            "confidence": round(float(max(probability)), 3),
            # Score on a 0-100 scale for easy interpretation
            "lead_score": round(float(probability[1]) * 100, 1)
        },
        "feature_importance": {
            name: round(float(imp), 3)
            for name, imp in sorted(
                zip(feature_names, importances),
                key=lambda x: x[1],
                reverse=True
            )
        }
    }

    print(f"\nLead Score: {ml_results['ml_scores']['lead_score']}/100")
    print(f"Classification: {ml_results['ml_scores']['classification']}")
    print(f"Confidence: {ml_results['ml_scores']['confidence']:.1%}")
    print(f"\nTop features driving this score:")
    for name, imp in list(ml_results['feature_importance'].items())[:3]:
        print(f"  - {name}: {imp:.3f}")
    print()

    return ml_results

## Stage 2: LLM Reasoning -- Decision Making

The LLM receives the ML scores as **EVIDENCE** and reasons about them.
It's not guessing -- it's analyzing real, computed data.

This is the magic of the ML + LLM pattern:
- The ML model did the math (fast, cheap, deterministic)
- The LLM does the thinking (flexible, contextual, natural language)

In [ ]:
def reason_about_lead_with_llm(ml_results: dict) -> dict:
    """
    Use GPT-4o to reason about the ML scoring results.

    Input: ml_results dict from Stage 1 (the ML model's output)
    Output: A recommendation dict with reasoning and action items

    The LLM sees the computed scores and feature importances, then
    provides human-readable reasoning and specific action recommendations.
    """

    print("=" * 60)
    print("STAGE 2: LLM Reasoning")
    print("=" * 60)

    ml_json = json.dumps(ml_results, indent=2)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": f"""You are a real estate lead management expert. A machine learning model
has scored a lead and provided the following data. Your job is to REASON
about this data and provide specific, actionable recommendations.

IMPORTANT: Base your reasoning ONLY on the data provided. Do not make up
information. Reference the specific scores and features in your analysis.

ML SCORING RESULTS:
{ml_json}

Analyze this data and return ONLY valid JSON in this format (no markdown,
no code fences):
{{
    "lead_assessment": "1-2 sentence assessment of this lead",
    "reasoning": "3-4 sentences explaining WHY the ML model scored them this way, referencing specific features and scores",
    "urgency": "high/medium/low",
    "recommended_actions": [
        "Specific action 1 with timing",
        "Specific action 2 with details",
        "Specific action 3 with context"
    ],
    "talking_points": [
        "What to mention when contacting this lead",
        "Another relevant talking point"
    ],
    "risk_factors": ["Any concerns about this lead"]
}}"""
            }
        ]
    )

    raw_output = response.choices[0].message.content

    try:
        recommendation = json.loads(raw_output)
    except json.JSONDecodeError:
        recommendation = {
            "lead_assessment": raw_output[:200],
            "reasoning": "Failed to parse structured response",
            "urgency": "medium",
            "recommended_actions": ["Review lead manually"],
            "talking_points": [],
            "risk_factors": ["LLM output parsing failed"]
        }

    print(f"\nAssessment: {recommendation.get('lead_assessment', 'N/A')}")
    print(f"Urgency: {recommendation.get('urgency', 'N/A')}")
    print(f"\nReasoning: {recommendation.get('reasoning', 'N/A')}")
    print()

    return recommendation

## Pipeline Orchestrator

Run the full ML + LLM pipeline.

**Flow:**
```
Lead features --> ML Model (scoring) --> LLM (reasoning) --> Recommendation
```

The ML model produces **EVIDENCE** (scores).
The LLM produces **REASONING** (explanations and actions).
Together they give you both the "what" and the "why".

In [ ]:
def run_ml_llm_pipeline(lead_features: dict) -> dict:
    """
    Run the full ML + LLM pipeline.

    Flow:
        Lead features --> ML Model (scoring) --> LLM (reasoning) --> Recommendation

    The ML model produces EVIDENCE (scores).
    The LLM produces REASONING (explanations and actions).
    Together they give you both the "what" and the "why".
    """

    print("\n" + "#" * 60)
    print("# ML + LLM PIPELINE START")
    print("#" * 60 + "\n")

    # First, train the ML model (in production, you'd load a pre-trained model)
    model, scaler = train_lead_scoring_model()
    print()

    # Stage 1: ML Scoring (fast, cheap, deterministic)
    ml_results = score_lead_with_ml(model, scaler, lead_features)

    # Stage 2: LLM Reasoning (flexible, contextual, natural language)
    recommendation = reason_about_lead_with_llm(ml_results)

    # Combine everything into a final output
    final_output = {
        "lead_features": lead_features,
        "ml_scores": ml_results["ml_scores"],
        "feature_importance": ml_results["feature_importance"],
        "recommendation": recommendation
    }

    print("#" * 60)
    print("# PIPELINE COMPLETE")
    print("#" * 60)

    print("\n" + "=" * 60)
    print("FINAL RECOMMENDATION:")
    print("=" * 60)
    print(json.dumps(final_output["recommendation"], indent=2))

    return final_output

## Example 1: Hot Lead

Score and analyze a lead with high engagement signals.

In [ ]:
print("\n--- Example 1: Hot Lead ---\n")
hot_lead = {
    "name": "Sarah K.",
    "days_since_last_contact": 2,
    "website_visits_this_week": 9,
    "email_open_rate": 0.72,
    "property_views": 15,
    "days_on_market": 21,
    "price_range_match": 0.85
}
result_hot = run_ml_llm_pipeline(hot_lead)

## Example 2: Cold Lead

Score and analyze a lead with low engagement signals.

In [ ]:
print("\n--- Example 2: Cold Lead ---\n")
cold_lead = {
    "name": "Mike T.",
    "days_since_last_contact": 45,
    "website_visits_this_week": 1,
    "email_open_rate": 0.12,
    "property_views": 2,
    "days_on_market": 120,
    "price_range_match": 0.25
}
result_cold = run_ml_llm_pipeline(cold_lead)

## Teaching Point: Compare the Two Results

The ML model gave us hard numbers: Sarah is 85/100, Mike is 12/100.
The LLM explained WHY and told us WHAT TO DO about it.

- **Without ML:** The LLM would have to guess at scores (unreliable).
- **Without LLM:** We'd have scores but no actionable recommendations.
- **Together:** We get reliable scores AND intelligent recommendations.

This is the power of the ML + LLM pipeline pattern.